Cell 1(markdown)
#### 01 | Raw 層資料稽核(M1 階段 A)

**目的**:</br>
逐欄稽核 `raw.bank_churners`，回答「每一欄能不能信、髒在哪」。</br>所有 M1-B 的清理決策(型別、Unknown、剔除欄)都以本稽核的輸出為依據。

**與 `sql/01_load_audit.sql` 的關係**:</br>
(1)兩個檔案都執行相同口徑的稽核(「指標定義」在兩個檔案中都相同)。</br>
(2)`.sql` 檔是正式版，功能是「純 psql 可重現」、「零依賴」；本 notebook 則是展示版，功能是「展示結果與判斷思路」。</br>
(3)兩者雖有重複的稽核 SQL ，但有各自不同的功能 (底線：清理邏輯就不能重複，只會放在 src/cleaning.py 中，確保清理邏輯更動時不需承擔多處需同步修改的出錯風險。稽核 SQL 寫完就固定，重複的維護成本極低)。兩者差異:notebook 用 Python 迴圈消除逐欄重複，sql 檔要用 UNION ALL 重複疊起 14 段指令。</br>
(4)notebook 檔中的資料庫欄位名是來自 cell 2 寫明的清單，所以用字串拼時不需擔心 SQL Injection。

**結構**(對應 .sql 檔 [A]–[H])：</br>
- A 主鍵 → B 類別分布 → C 可轉型稽核 → D 數值範圍 → E/F 欄位關係實證 → G 跨欄邏輯矛盾 → H 基準流失率 → 稽核結論總表<br/>
- 每個稽核區塊的設計：code cell 產生證據，判讀 cell 完成「證據 → 判斷 → 決策」這三步驟

In [1]:
# Cell 2(code)|環境設定

import sys
import pathlib
sys.path.append(str(pathlib.Path.cwd().parent))  # 將專案根目錄加入模組搜尋路徑：目前工作目錄是在 notebooks/，加入這行才能回到 churnlens/ 找到src/ 並引入旗下的 db.py 的 get_engine()。

import pandas as pd
from src.db import get_engine

engine = get_engine()

# 在此將要稽核的 20 個欄位寫死成兩個清單，並依「類別、數值」分類。[B][C][D] 三個迴圈皆引用此處。
# 「個別欄位歸屬哪個分類」的設定只存在這個 cell，增減稽核欄位只改這裡，方便日後維護時不容易漏。(sql檔就要到每區塊一個個修改，有漏改風險)
# 欄位清單總共 20 個 (資料集共 23 個欄位)：清單不含 clientnum(由 [A] 單獨稽核)與 nb_classifier_prob_1/2(M0 已判定剔除，所以不稽核)

CATEGORICAL_COLS = [
    "attrition_flag", "gender", "education_level",
    "marital_status", "income_category", "card_category",
]
NUMERIC_COLS = [
    "customer_age", "dependent_count", "months_on_book",
    "total_relationship_count", "months_inactive_12_mon",
    "contacts_count_12_mon", "credit_limit", "total_revolving_bal",
    "avg_open_to_buy", "total_amt_chng_q4_q1", "total_trans_amt",
    "total_trans_ct", "total_ct_chng_q4_q1", "avg_utilization_ratio",
]

Cell 3 (markdown)
#### [A] 總筆數與主鍵唯一性
(1) 先驗「一列一客戶」的結構假設，再驗欄位內容：此假設是後續所有計數與比率的計算前提，clientnum 若有重複，同一客戶被重複計數，客戶數與流失率都會失真。<br/>
(2) 失真(distortion)：直接後果是數字失真(分母多算了重複的人，流失率可能變成別的數)；連帶後果是統計意義失真(因為指標意義不小心變了但沒人發現，把錯的結論當真：同一客戶被重複計數，『流失率』的語意會從客戶層悄悄變成列層)<br/>

**預期**：<br/>
total_rows = 10127，distinct_clients = 10127，null_clientnum = 0<br/>
|      | total_rows | distinct_clients | null_clientnum |
| :--- | :--- | :--- | :--- |
| 0    | 10127 | 10127 | 0 |

In [2]:
# Cell 4 (code)
# pd.read_sql:SQL 原文送 DB 執行(聚合在 DB 端完成)，Python 只收結果並包成 DataFrame (細節：Python 跟 engine 要一條 DB 連線，將 SQL 字串轉送給 PostgreSQL。再來，由 DB 解析 SQL、掃表、做聚合計算，把計算結果回傳給 Python。然後，pandas 把這幾個數字加上欄名(來自 SQL 的 AS 別名)、加上列索引 (最左邊那個 0，是 pandas 自動加的)，組成 DataFrame，最後歸還 DB 連線)
audit_a = pd.read_sql("""
    SELECT COUNT(*)                      AS total_rows,
           COUNT(DISTINCT clientnum)     AS distinct_clients,
           SUM((clientnum IS NULL)::int) AS null_clientnum
    FROM raw.bank_churners
""", engine)
audit_a

,total_rows,distinct_clients,null_clientnum
0,10127,10127,0


Cell 5(markdown)|判讀

**判讀** (預期 → 實測 → 決策意涵) 

- 預期：「一列 = 一個客戶」成立，clientnum 可當主鍵。
- 實測：___(1.跑完填真實數字。不如預期時，加填：重複列的明細 + 處置方式，例如：完全重複 → 去重保留一列；欄位值衝突 → 訂規則擇一或整筆剔除。2.留白原則：證據必須來自執行，不是記憶或期望：實測結果先留白才能強迫「觀察」這個動作真實發生)
- 決策意涵：成立 → clean 層以 clientnum 為主鍵；不成立 → M1-B 需先訂去重規則。

Cell 6(markdown)
#### [B] 類別欄分布:值域(有哪些值) + 筆數(每個值各出現幾筆) + 占比(每個值的筆數/全表筆數)
1.核心決策點:(education_level、marital_status、income_category 三欄含 Unknown 值)，本區實測各欄位中 Unknown 值的實際占比(該欄位的 Unknown 筆數/全表筆數 10,127)，決定 M1-B 對 Unknown 的處置方式：(1)把 Unknown 列保留並將 Unknown 作為一種獨立類別、自成一組；或 (2)剔除 (排除含 Unknown 的整列。代價：損失樣本、引入「不願透露者」偏誤(此族群可能有特定行為特質，剔除後樣本不再代表全體))。<br/>
2.決策對象是「Unknown 這個值，以及帶著這個值的該列」。<br/>
3.實測數字直接決定清理規則：例如，Unknown 的占比 1% 和 15%，可能對應不同的清理原則。<br/>
4.語法:`SUM(COUNT(*)) OVER ()` 於分組後取得全表總數，使每組能算占比。

**預期**：<br/>
|      |         col        |        val         |  n    |  pct   |
| :--- | :--- | :--- | :--- |:--- |
| 0    |    attrition_flag   | Existing Customer | 8500  | 83.93  |
| 1    |    attrition_flag   | Attrited Customer | 1627  | 16.07  |
| 2    |    gender           | F                 | 5358  | 52.91  |
| 3    |    gender           | M                 | 4769  | 47.09  |
| 4    |    education_level  | Graduate          | 3128  | 30.89  |
| ...  |         ...         |       ...         |  ...  |  ...   |

In [3]:
# Cell 7(code)
# 對六個類別欄，各跑一次同樣的值分布查詢(查詢該欄的值的分布：有哪些值、每個值幾筆、占比多少)。得出：六張小表，最後疊成一張大表

# 先建一個空 list，等等裝六張 DataFrame
frames = [] 
# 遍歷 cell 2 那份寫死的清單：f-string 組出每一圈的 SQL → 送去資料庫執行(pd.read_sql，在 DB 聚合) → 結果 DataFrame 塞進 list(append)
# '{col}' AS col (作為標籤):帶引號，替換後是 'gender' (SQL 字串常量(寫死在 SQL 裡的固定值，非欄名、非變數)，每列都原樣輸出 gender 這個字作為標籤(同 .sql 檔貼標籤的寫法)。{col} AS val (真的去查 gender 欄位的值):不帶引號，替換後是 gender (SQL 識別字，真的去查這個欄位)。引號是 SQL 區分「這是一段文字」vs「這是一個欄位/表的名字」的方式。
# 先以該欄的值分組(GROUP BY)，然後算各組筆數(n)；再用 window function 取全表總數，算各組的占比；最後依 n 由大到小排序。最終得出該圈的：標籤(col)、各組名(值域)(val)、各組筆數(n)、各組佔比(pct)。
# 100.0 * 要寫在除法前面：COUNT(*) 是整數，整數 / 整數會無條件捨去小數 (8500/10127=0)。100.0 帶小數點、屬 numeric 型別，先乘它讓分子提升為 numeric，除法才保得住小數。若寫成「先除再乘 100.0」，除法當下仍是整數相除，已捨去的小數救不回來。

for col in CATEGORICAL_COLS:
    frames.append(pd.read_sql(f"""
        SELECT '{col}' AS col, {col} AS val, COUNT(*) AS n,
               ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS pct
        FROM raw.bank_churners
        GROUP BY {col} ORDER BY n DESC
    """, engine))
# 迴圈跑完後，frames 是一個裝著六張 DataFrame 的 list：第一個 2 列(attrition_flag 兩種值)、第二張 2 列(gender)、第三張 7 列(education_level)...各表欄位結構相同(col / val / n / pct)。

# 把 frames 這個 list 裡的每張小表上下疊起來成一張大表(同 SQL 的 UNION ALL概念，但 pandas 用 pd.concat 疊表)，產出一張 25 列上下的總表 (2+2+7+4+6+4)
# ignore_index=True：重編「列索引」。若沒加，每張小表自帶的索引原樣保留(0、1、0、1、0、1、2、3)重複又混亂；加了後會從0連續編到底 (0-24)。若有「疊完產出一張新表」的需求，就要這樣加。
audit_b = pd.concat(frames, ignore_index=True)
audit_b

# 對比 .sql 檔的六段查詢，sql 檔若要改 audit 的口徑(指標定義)，要分別改六個地方。這裡是「寫一次邏輯、欄位清單控制範圍」，「用一段邏輯處理 N 個對象」較高效率。寫法：「清單 + 迴圈 + concat」。
# 本處寫法 trade-off：audit 的情境選「可讀性優先，效率次之」。可讀性：本案表小，六次查詢每次皆毫秒級完成，效率損失無感，故取可讀性，for 迴圈可讀性高。可優化(提高效率)：這寫法是「對資料庫發出六次獨立查詢、掃了六次表」，更高效率作法：組一條 UNION ALL 大查詢一次送出，或用一次掃描的 unpivot 技巧。

,col,val,n,pct
0,attrition_flag,Existing Customer,8500,83.93
1,attrition_flag,Attrited Customer,1627,16.07
2,gender,F,5358,52.91
3,gender,M,4769,47.09
4,education_level,Graduate,3128,30.89
5,education_level,High School,2013,19.88
6,education_level,Unknown,1519,15.00
7,education_level,Uneducated,1487,14.68
8,education_level,College,1013,10.00
9,education_level,Post-Graduate,516,5.10


Cell 8(markdown)|判讀

**判讀**

- 各欄的值域是否乾淨 (例如：無錯字變體、無空字串、無預期外的值) ➡ 實測：_____
- Unknown 占比：education_level _____ %|marital_status _____ %|income_category _____ %
- 樣本量警戒：有沒有占比極小的類別(例如：< 2 %)？ ➡ _____
  (M2 切片與 M3 檢定時，樣本數過小的類別，其切片格的比率受隨機波動主導，結論不可信。先在這裡標記，這是「稽核支持決策」的紀律。[B]支持的不只是 Unknown 的處置決策，還包括 M2/M3 的分析範圍決策)
- 決策意涵：Unknown 占比若達雙位數，執行剔除 = 損失上千樣本 + 引入「不願透露者」偏誤 → 傾向保留為獨立類別 (最終於 M1-B 拍板，並記入決策日誌)。

Cell 9(markdown)
#### [C] 數值欄可轉型稽核 + NULL 檢查
TEXT 層特有的檢查關卡：在這區先檢查「每個該是數字的欄，是否每一格都通過數值格式正則」。正則以「::numeric 收不收」為設計標準，通過即保證後續 [D]–[G] 的臨時轉型與 M1-B 的永久轉型都安全。最終型別於 M1-B 依各欄語意選定，例如整數欄用 int。<br/>
補充：<br/>
(1)以 ::numeric 為通用標準是因為 numeric 同時容納整數與小數，一種寫法驗所有欄。<br/>
(2)正則 `^\d+(\.\d+)?$` 只放行正整數/正小數；NULL 不在正則守備範圍內，所以另行查詢。<br/>
與.sql 檔的寫法差異：.sql 檔查 NULL 要寫 14 行，這裡則在既有迴圈裡加一行 SQL 就能同時完成兩項稽核。「一段邏輯 × N 個對象」的架構在擴充時有優勢：加檢查項 = 加一行，不是加 14 行。<br/>
**預期**：<br/>
預期輸出「14 列 × 3 欄」，bad_rows 與 null_rows 應全 0 (此資料集以 'Unknown' 表缺失，應無真 NULL。但還是在這邊做一次檢查是否無真 NULL 作為紀錄)。<br/>
|      |          col        |  bad_rows  |  null_rows   |  
| :--- | :--- | :--- | :--- |
| 0    |    customer_age                 | 0 | 0 | 
| 1    |    dependent_count              | 0 | 0 |
| 2    |    months_on_book               | 0 | 0 |
| 3    |    total_relationship_count     | 0 | 0 |
| 4    |    months_inactive_12_mon       | 0 | 0 |
| 5    |    contacts_count_12_mon        | 0 | 0 | 
| 6    |    credit_limit                 | 0 | 0 | 
| 7    |    total_revolving_bal          | 0 | 0 | 
| 8    |    avg_open_to_buy              | 0 | 0 | 
| 9    |    total_amt_chng_q4_q1         | 0 | 0 | 
| 10   |    total_trans_amt              | 0 | 0 | 
| 11   |    total_trans_ct               | 0 | 0 | 
| 12   |    total_ct_chng_q4_q1          | 0 | 0 | 
| 13   |    avg_utilization_ratio        | 0 | 0 | 

In [4]:
# Cell 10(code)
# 先建一個空 list，等等裝十四張 DataFrame
# 每圈的 SQL 對一個欄位同時做兩項檢查：壞格式計數(bad_rows)+ NULL 計數(null_rows)
# 用 pd.concat 將 14 張一列的小表疊成 14 列總表 (同 Cell 7)
rows = []
for col in NUMERIC_COLS:
    rows.append(pd.read_sql(f"""
        SELECT '{col}' AS col,
               SUM(({col} !~ '^\\d+(\\.\\d+)?$')::int) AS bad_rows,
               SUM(({col} IS NULL)::int)               AS null_rows
        FROM raw.bank_churners
    """, engine))

audit_c = pd.concat(rows, ignore_index=True)
audit_c

# 與 .sql 檔的輸出樣貌差異:.sql 檔一個查詢塞 14 個 SUM，結果是 1 列 × 14 欄(寬表)，欄名承載「哪個欄的檢查」; 本 cell 迴圈每圈查一欄、產一列，疊成 14 列 × 3 欄(長表)，欄名是格子內的資料內容。迴圈產長表：迴圈一次處理一個對象(欄位)，每圈產出「關於該對象的一列」。長表較好讀、好用：直向可一眼掃到底，且可直接操作(如 audit_c[audit_c.bad_rows > 0] 篩出問題欄，寬表做不到)。
# 因為 Python 吃一層反斜線，所以預先多寫一層反斜線；而 .sql 檔無 Python 轉手，單反斜線即可：正則本體是 ^\d+(\.\d+)?$ ，與 .sql 檔相同。但此 cell 的 SQL 語法位在 Python 字串裡，而反斜線是 Python 字串的逃逸字元，所以 Python 會先吃掉一層反斜線，因此寫 \\d,送到 PostgreSQL 手上會還原成 \d。
# 排序從 SQL 層(ORDER BY)移到 Python 層(清單書寫順序)：每圈查詢無 GROUP BY。因為聚合函式把整張表當一個組，一萬列餵進 SUM 會吐一個總數，只回 1 列。只有一列就無排序的情境，所以也不需要 ORDER BY。最終 14 列的順序不是 SQL 排的，是 Python 排的：迴圈按 NUMERIC_COLS 清單的順序跑，然後 concat 按 list 順序疊。

,col,bad_rows,null_rows
0,customer_age,0,0
1,dependent_count,0,0
2,months_on_book,0,0
3,total_relationship_count,0,0
4,months_inactive_12_mon,0,0
5,contacts_count_12_mon,0,0
6,credit_limit,0,0
7,total_revolving_bal,0,0
8,avg_open_to_buy,0,0
9,total_amt_chng_q4_q1,0,0


Cell 11(markdown)|判讀

**判讀**
- ➡ 實測：bad_rows / null_rows 是否全 0？ _____<br/>
- 若任一欄 > 0：撈明細(`SELECT {col}(代入真實欄名) FROM raw.bank_churners WHERE {col}(代入真實欄名) !~ '^\d+(\.\d+)?$' LIMIT 10`)看髒值的模式並記錄髒值的長相(例如：記錄「credit_limit 髒值 37 筆：千分位逗號 30、空字串 7」)，
  然後依據不同髒值模式(例如：逗號、空字串)訂定不同資料清洗規則並附理由。例如：(1)修復逗號並轉型：因為逗號是格式問題不是資料問題，值本身可信；(2)把空字串轉 NULL：因為空字串語意上就是缺失，轉成 NULL 讓它被正確地當缺失處理，而不是轉型時壞掉。
- M1-B 先訂好資料清洗規則後，之後才可執行「安全轉型別」。「訂規則」是： 把「人看樣本得出的模式判斷」翻譯成「機器可執行的轉換程式碼」。在這階段先做記錄是為了確保這個翻譯可被稽核。<br/>
- 決策意涵：全 0 → M1-B 可對 14 欄直接安全轉型，不需前置的資料清洗流程。<br/>

Cell 12(markdown)
#### [D] 對數值欄位的資料範圍做摘要(min / max / avg)，判斷是否具業務合理性
輸出後對照 `01_load_audit.sql` 末尾的 [D-附錄] 業務合理性邊界表，判斷各個欄位資料的極值與平均值是否合理。
**預期**：<br/>
預期輸出(14 列 × 4 欄，已按 col 字母排序) <br/>
以下預期數值來源：非「本專案實測值」，而是參考「社群對本公開資料集的相關描述統計」而得出的「合理預期數值」。此預期數值在本專案的參考地位同 corr≈0.97 (都是社群流傳的數值，僅供本專案對照)。
本專案各欄位的實際數值一律以「下方 code 實測」為準；若與本處「合理預期數值」落差過大，需記錄並判斷原因及處置決策(此即為「資料稽核」階段的意義之一)。

|      |               col              |  min  |  max |  avg  |
| :--- | :--- | :--- | :--- |:--- |
| 0    |       avg_open_to_buy          |   3.0    |   34516.0   | 7469.14 | 
| 1    |       avg_utilization_ratio    |   0.0    |   0.999     | 0.27    | 
| 2    |       contacts_count_12_mon    |   0.0    |   6.0       | 2.46    | 
| 3    |       credit_limit             |   1438.3 |   34516.0   | 8631.95 | 
| 4    |       customer_age             |   26.0   |   73.0      | 46.33   | 
| 5    |       dependent_count          |   0.0    |   5.0       | 2.35    | 
| 6    |       months_inactive_12_mon   |   0.0    |   6.0       | 2.34    | 
| 7    |       months_on_book           |   13.0   |   56.0      | 35.93   | 
| 8    |       total_amt_chng_q4_q1     |   0.0    |   3.4       | 0.76    | 
| 9    |       total_ct_chng_q4_q1      |   0.0    |   3.7       | 0.71    | 
| 10   |       total_relationship_count |   1.0    |   6.0       | 3.81    | 
| 11   |       total_revolving_bal      |   0.0    |   2517.0    | 1162.81 | 
| 12   |       total_trans_amt          |   510.0  |   18484.0   | 4404.09 | 
| 13   |       total_trans_ct           |   10.0   |   139.0     | 64.86   | 

In [5]:
# Cell 13(code)
# 每一圈查出一列資料(欄位名、min、max、avg)並以dataframe形式裝進rows列表。
# concat 將列表中14個dataframe疊成一張大表並加入列索引，然後用依col欄位內的值(eg.credit_limit)按字母序重新排列，最後依新排序重編列索引 (sort_values 後接 reset_index(drop=True) 是 pandas 慣用句，類似SQL的ORDER BY 但在 pandas 要兩個動作才等價)
# Pandas 的 method chaining：利用「每個操作回傳新物件」的特性，把多個依序的操作寫成一條連續呼叫鏈，最後一個操作完的結果才被存進最終變數 (中間產生的物件都不會用到，省下為中間物件命名的多餘動作。) 適用「方法鏈」的情境：操作在三四步以內、每步可讀性高；適合「把每個方法拆開分別把中間結果存入不同變數的」的情境：操作更多步、有步驟需中途檢查。

rows = []
for col in NUMERIC_COLS:
    rows.append(pd.read_sql(f"""
        SELECT '{col}' AS col,
               MIN({col}::numeric)           AS min,
               MAX({col}::numeric)           AS max,
               ROUND(AVG({col}::numeric), 2) AS avg
        FROM raw.bank_churners
    """, engine))

audit_d = pd.concat(rows, ignore_index=True).sort_values("col").reset_index(drop=True)
audit_d

,col,min,max,avg
0,avg_open_to_buy,3.0,34516.000,7469.14
1,avg_utilization_ratio,0.0,0.999,0.27
2,contacts_count_12_mon,0.0,6.000,2.46
3,credit_limit,1438.3,34516.000,8631.95
4,customer_age,26.0,73.000,46.33
5,dependent_count,0.0,5.000,2.35
6,months_inactive_12_mon,0.0,6.000,2.34
7,months_on_book,13.0,56.000,35.93
8,total_amt_chng_q4_q1,0.0,3.397,0.76
9,total_ct_chng_q4_q1,0.0,3.714,0.71


Cell 14(markdown)|判讀

**判讀** (對照 [D-附錄] 邊界表)
- ➡ 實測中「怪異不合理」的列：___ (逐列迅速判斷：年齡是 18–100 嗎？ utilization 是 0–1 嗎？ chng_q4_q1 變化比有無異常倍數？)<br/>
- 特別注意右偏(right-skewed)欄位 (credit_limit、total_trans_amt)：max 遠大於 avg 是金額類資料的常態，不是髒值。
    - (1)判斷是否為髒資料的標準不是「離平均遠不遠」，而是「是否有業務上的可能」。例如：credit_limit 的 max = 34,516 是可能的，所以可留下。反之，年齡 250 歲不可能，所以需處置(剔除或標記)。
    - (2)把「自然、可能的資料右偏」當髒值清掉，比不清理更糟：因為清掉反而「刪掉最有價值的高端客群 (信用額度較高的客群)」。
    - (3)「資料右偏」如何影響下游的資料處理(如 M2、M3)：例如，若只有極少數客群是極大值，則 M2 分箱不能等寬分箱，而要改用「等量分箱(如 NTILE 函數)」(因為等寬切的箱界由 (max−min)/箱數 均分，極大的 max 會把每箱範圍撐得很寬，例如第一箱就是0-6900，結果 90% 樣本都擠進第一箱)；M3 比較平均值時，平均數會被長尾側的極端大值拉高，所以須留意極端值影響。<br/>
- 決策意涵：範圍內 → 不做極端值處理；範圍外 → M1-B 撈明細時定資料處置規則：剔除(適用於「這個錯誤污染整列的可信度」)或標記(適用於「值可疑但列還有價值」)。

Cell 15(markdown)
#### [E] 陷阱二實證：avg_open_to_buy = credit_limit − total_revolving_bal？
驗證目標：實證「剩餘可用額度 = 信用額度 − 循環欠款(本金)」這條衍生關係是否成立。<br/>

**容許誤差 0.01(防上游 float 殘差，追求實質相等)**:<br/>
(1) 防「上游 float 殘差」：資料在上游的生產鏈(eg.銀行系統計算、匯出、CSV 轉存)若經過浮點數運算，可能留下微小誤差(如 9779 存成 9778.9999...)。殘差是上游留下的，非本專案資料庫端產生(因為此處用::numeric精確運算，環節乾淨，排除「此處的查詢計算發生誤差」的可能性)。<br/>
(2) 因此比較採「實質相等」(差距小到業務上無意義即視為相等)而非「位元相等」(若用位元完全相等來嚴格比較，極微小的浮點殘差也會製造假 mismatch)。<br/>
(3) 0.01 的定法：取「業務最小有意義單位」：金額欄的最小單位是「一美分(0.01元)」，差距 < 0.01 在業務上不存在，≥ 0.01 才是真的有金額落差。此為實務上金額類「容許誤差極限」的通用定法。<br/>

**預期**:
預期輸出 (1 列 × 1 欄；mismatch_rows = 0 → 等式實證成立)：
|      |    mismatch_rows     |  
| :--- | :--- | 
| 0    |   0   | 

In [6]:
# Cell 16(code)

# WHERE 語意:「留下 |實際值 − 理論值| 超過 0.01 的列」，COUNT 數這種列有幾筆。
# 寫法選擇：單一條件用 WHERE+COUNT(先篩列再數)最直白；與 [C][G] 的 SUM((條件)::int)(全列掃、條件計數)都是先篩列再數，但後者適用「多個篩選條件(eg.[C]區檢查 資料通過正則 且 資料不為 NULL)並排輸出一列」的情境。
# mismatch_rows： =0 → 實證成立(判讀走分支 1)； >0 → 需追查資料哪裡出問題 (走分支 2)。
audit_e = pd.read_sql("""
    SELECT COUNT(*) AS mismatch_rows
    FROM raw.bank_churners
    WHERE ABS(avg_open_to_buy::numeric
              - (credit_limit::numeric - total_revolving_bal::numeric)) > 0.01
""", engine)
audit_e

,mismatch_rows
0,0


Cell 17(markdown)|判讀
**判讀**
- ➡ 實測: _____
- **(1) = 0 → 等式實證成立，avg_open_to_buy 是線性衍生欄，全案排除此欄。**
    - 衍生欄：值可由其他欄算出(此處 A = B − C 即為線性運算)，即 A 不含任何新資訊。
    - 排除理由(M4 影響最大)：三欄同進模型會完全共線(perfect multicollinearity)：A 恆等於 B−C，同一預測效果有無限多種係數分法，係數變得任意而失去解釋意義；本案 M4 旨在建立「可解釋模型」，所以需排除此衍生欄。
    - 排除方式：建模特徵的清單不放 A + 決策日誌記明理由。M2 切片及 M5 分群同理(「依 A 切」同「依 B−C 切」，不必重複)。
    - 三欄留哪兩欄：刪 A(衍生欄、無資訊損失)；保留的 B、C 中，total_revolving_bal(C) 是隨客戶行為每期變動的訊號(欠/還的變化直接反映流失前兆，敏感度最高)；credit_limit(B) 是銀行授信結果(客戶動不了、短期固定，訊號弱但提供規模脈絡)。留下的 B 和 C 欄中，C 較 B 更具流失敏感度。
- **(2) > 0 → 撈 mismatch 明細，觀察 diff 的分布模式來判定:**
    - 撈法：SELECT 特定欄位 + diff 欄
    - ```sql
      SELECT 
         clientnum, credit_limit, total_revolving_bal, avg_open_to_buy,
         avg_open_to_buy::numeric - (credit_limit::numeric - total_revolving_bal::numeric) AS diff
      FROM raw.bank_churners
      WHERE ABS(avg_open_to_buy::numeric - (credit_limit::numeric - total_revolving_bal::numeric)) > 0.01
      LIMIT 20
      ```
    - (a) diff 都是同方向的小額固定值(例如全部多 25 元) → 像「有隱含資訊」：例如，真實公式可能含文件未載的項如「保留額度」(額度 − 欠款 − 某種保留額) → 這種情境下，A 反而含獨立資訊，值得保留深入調查。
    - (b) diff 忽大忽小、正負無規律，且 B、C 各自已通過 [C][D][G] 單欄稽核 → 像「A 的計算環節壞了」：
      A 是三欄中唯一經過計算的欄，B/C 是直接記錄的原始值；原始值通過稽核但最終成品對不上正常結果，問題高機率出在「計算這個加工環節」，所以 
      → 剔除 A **欄**(非剔除列。原則：當錯誤沿「欄」分布 → 處置欄；當錯誤沿「列」分布(同列多欄皆有問題)→ 才處置列)。
    - (c) 僅極少數列 mismatch、其餘完美吻合 → 衍生關係整體成立，一樣刪 A 欄；mismatch 列則另行標記。
    - (d) 以上為證據指向的推斷，並非嚴謹證明。實務情形應「向上游確認資料生成過程是否有問題」；本案為公開資料集，上游不可考，所以僅「推斷 + 記錄推理過程」(原則：記錄完整推理過程，若之後結論有問題，也可回溯檢驗推理過程是否有誤；若只記錄結論、沒記錄推理過程，結論若對可能只是運氣好、但無從檢驗過程，結論若錯就更沒有紀錄能檢驗過程中發生什麼事、進而查錯)。

Cell 18(markdown)
#### [F] 陷阱三實證：age 與 tenure 的 Pearson 相關係數

- 社群流傳本資料集有「客戶年齡與帳戶年齡的相關係數高達0.97」之非自然陷阱，流傳的相關係數數字從 0.79 到 0.97 都有 → 一律視為「待驗證內容」，以下方實測為準，作為後續 M1-B 清洗、M2 分析、M4 建模的特徵決策依據。
- 若實測值高達 0.97，證實「帳戶年齡」非獨立變數，而只是「客戶年齡」等比例連動的影子特徵 (Shadow Feature)(資料集可能藏有「全體客戶幾乎都在同歲辦卡」的荒謬特徵)，表示：帳戶年齡這個欄位沒有帶入任何獨立新資訊，它只是隨「客戶年齡」等比例放大或縮小。

**預期**:
預期輸出 (1 列 × 1 欄；社群流傳 0.79 到 0.97，以實測為準)：
|      |    corr_age_tenure     |  
| :--- | :--- | 
| 0    |   0.9xxx (← 實際數字待實測，不預填)   | 

In [7]:
# Cell 19(code)
# PostgreSQL 中，integer、double precision、numeric 是平行的三種數值型別
# CORR 函式的參數只接受數值(包含：integer、double precision、numeric)，所以CORR()內兩個參數都要先從 TEXT 轉 numeric (原 TEXT 資料某些可能是小數資料、但 integer 無法表示小數；double precision 可能有浮點數問題造成精度誤差，所以通常選擇轉換成 numeric)
# ROUND 函式(兩參數版本)的參數只接受 numeric 作為第一個參數 (單參數版的 ROUND 函式能接受 double precision 和 numeric)，但 CORR() 算完後回傳的型別是 double precision，所以要先把 double precision 轉成 numeric，才能用 ROUND() 函式取到小數點後四位。
audit_f = pd.read_sql("""
    SELECT ROUND(CORR(customer_age::numeric, months_on_book::numeric)::numeric, 4)
           AS corr_age_tenure
    FROM raw.bank_churners
""", engine)
audit_f

,corr_age_tenure
0,0.7889


Cell 20(markdown)|判讀
**判讀**
- ➡ 實測：corr = _____
- 分級決策意涵(基本邏輯：兩欄若高度相關，M4 建模時模型會係數不穩、分不清誰在影響流失 → 需對這兩個資料欄位二擇一；M2 按 tenure 切片時，tenure 組內容大部分同特定年齡組內容，所以觀察流失差異時會無法判斷是「帳齡造成」還是「年齡造成」 → 發生「混淆(confounded)」):
  - ≥ 0.95：tenure 幾乎是 age 的影子特徵、未帶入獨立資訊(真實世界不可能人人同歲辦卡，這種相關係數表示「上游資料很可能是由人工抽樣演算法合成而造成問題」)→ M4 必須兩欄位二擇一；M2 的 tenure 分析結論需大幅保留(混淆最嚴重，不可在這種情境下依據資料下因果結論)。
  - 0.7 – 0.95：高度相關但 tenure 仍有部分獨立變異 → M4 仍建議兩欄位二擇一(此區間係數仍不穩)；M2 可謹慎使用 tenure 切片，但每次判讀時須附註「帳齡與年齡混淆」的限制情形，將「能說的」和「不能說的」劃分清楚。
  - < 0.7：社群流傳的數字被本案實測推翻 → 記入日誌作為分析素材；表示「兩欄各有獨立資訊，可並存，M4 不需強制二擇一」。
- 無論落在哪個分級，將「以 SQL 實測取代文件或社群宣稱」這件事寫進清理紀錄。
- 假設先行：預先把每個分支的行動寫好，實測出來直接依據「框定好的不同假設情境」做相對應的不同行動，判斷才會是中立的。這個做法也防止「看到數字才找理由」的事後合理化。

- 補充：各分級的界線設定是根據以下判斷依據：
  - (1)0.95 取自建模慣例的共線警戒(VIF≈10 換算)、同時是「自然人群到不了」的常識線(自然人群裡，age 和 tenure 的相關頂多中高(年紀大的確平均辦卡久,但每個人辦卡年齡差很多),真實信用卡組合跑出 0.95+ 幾乎不可能)；
  - (2)0.7 取自統計慣例的強相關下限 (統計學上相關係數 0.7 算強相關)、同時落在社群流傳範圍(0.79-0.97)之外，能明確承載「外部宣稱的數字被推翻」的解釋語意。
  - (3)換一個資料集或換一個問題，這兩個數字就要重新思考設定。因為：分級原則可複用，但設定的數字門檻必須依不同情境重新論證。

Cell 21(markdown)
#### [G] 跨欄位邏輯矛盾檢查
無法在單格資料中檢查出來的髒資料，必須用「跨欄位組合檢查」來判斷。本區設定六條「業務上不可能發生」的組合，以此為規則做檢查，並**預期全 0**。<br/>

預期輸出(1 列 × 6 欄；全 0 = 無跨欄位的矛盾):
|      | ct0_but_amt | amt0_but_ct | util_over_1 | age_out_of_range | inactive_over_12 | revolving_over_limit |
| :--- | :--- | :--- | :--- | :--- | :--- | :--- |
| 0    |      0      |      0      |      0      |        0         |        0         |          0           |

In [8]:
# Cell 22(code)

# 六條檢查都是數值比較(= 0、> 1、< 18、欄與欄比大小)，而 raw 層的欄位都是 TEXT，所以要先轉成 numeric，再進行「數值的比較」。(若沒轉成numeric而是用字串來逐字元比較，會直接用字典序比而給出錯誤的答案，而且不會有語法報錯，而是直接安靜地給錯的答案導致判斷錯誤)
# 這區能這樣逐格轉型別，是因為 C 區已經有先把 14 欄都驗證過可以安全轉型 (C 區已檢查這 14 欄都是正整數、正小數、非 NULL。若有資料不符合前述條件(eg.字串)，用::numeric 做轉型會報錯)
# 語法：
# (1) 準備：解析 SELECT 語句，資料庫建立6個累積器(每個累積器的初始值都為0，例如：ct0_but_amt=0)。
# (2) 掃描 (一列一列掃，一口氣一次掃完全表)：
# a. 掃第一列客戶：取出該列相關欄位的值(本區要取7個欄位)，六個條件式各算一次(轉型 → 比較 → 得出布林值 → 用::int轉成1或0)，第一列客戶得出六個 0，每個0分別加入不同累積器 (累積器1(ct0_but_amt) += 0， 累積器2(amt0_but_ct) += 0， ...， 累積器6 += 0，此刻六個累積器仍是 0,0,0,0,0,0)。SUM 的行為是「作為累積器(如 ct0_but_amt)本人，每收到一個該列客戶的數字(0或1)就往該累積器內部加入這個數值，一列一列地累積往上加」
# b. 掃第二列...第n列客戶：同 a 步驟，各自加進不同累積器
# c. 掃第 8442 列：同上，但掃到 util=1.02，1.02 > 1? → true → 1，累積器3 += 1，累積器3 此刻變成 1，其他累積器不動。
# d. 掃第 10127 列 (最後一列)：所有累積器都加完
# (3) 收尾：掃描結束，這時 SUM 才「完成」(SUM 就是「累積器本人」)。最後，六個累積器的最終值 = 六個 SUM 的結果。把六個 SUM 結果組成一列結果，掛上各自的 AS 別名。
# (4) 資料庫回傳、pandas 處理：PG 資料庫把這一列回傳給 pandas → pandas 把這一列結果包成 DataFrame (1 列 × 6 欄，欄名=別名，列索引=0))，存進 audit_g。
audit_g = pd.read_sql("""
    SELECT
      SUM((total_trans_ct::numeric = 0 AND total_trans_amt::numeric > 0)::int) AS ct0_but_amt,
      SUM((total_trans_amt::numeric = 0 AND total_trans_ct::numeric > 0)::int) AS amt0_but_ct,
      SUM((avg_utilization_ratio::numeric > 1)::int)                           AS util_over_1,
      SUM((customer_age::numeric < 18 OR customer_age::numeric > 100)::int)    AS age_out_of_range,
      SUM((months_inactive_12_mon::numeric > 12)::int)                         AS inactive_over_12,
      SUM((total_revolving_bal::numeric > credit_limit::numeric)::int)         AS revolving_over_limit
    FROM raw.bank_churners
""", engine)
audit_g

,ct0_but_amt,amt0_but_ct,util_over_1,age_out_of_range,inactive_over_12,revolving_over_limit
0,0,0,0,0,0,0


Cell 23(markdown)|判讀

**判讀**
- ➡ 實測:六項各為 _____
- 全 0 → 無跨欄位組合的矛盾，M1-B 不需另外訂定清洗規則來處理。
- 任一 >0 → M1-B 階段要撈明細判定(如 util_over_1 若 > 0，則須區分「合理的小額授權超刷」vs「系統資料錯誤的問題資料」)再定處置(剔除或標記)。

Cell 24(markdown)
#### [H] 基準流失率：在 M1 階段(Day-One) 確立流失率的指標定義(口徑)
1.流失率 = 「attrition_flag = 'Attrited Customer' 」的客戶數」/ 全體客戶數。<br/>
2.此後每個階段的任何工具，若算出的基準流失率 ≠ 本數字，就代表該工具的計算邏輯可能有誤。<br/>
3.解釋：本區的意義是訂定本案的「基準線（Baseline）」(流失率的計算公式)。因為：
- 依本案問題定義書(docs/problem_definition.md)」，本案以「季度客戶留存率(= 1 - 流失率)」為全案北極星指標（核心業務目標）。本段 SQL 將「流失率這個指標的定義」確立在「本專案起點(Day-One) (M1 階段)」：流失率 = 「attrition_flag = 'Attrited Customer' 的客戶數」 / 全體客戶數。後續 M2 切片、M4 預測標籤、儀表板數字，都要以此公式為依歸，不能再任意更改這個指標的定義。

**預期**: 
預期輸出(1 列 × 1 欄；必須 = 16.07，與 M0 smoke test 的計算結果相同，對齊 M0 smoke test)：
|      | churn_rate_pct |
| :--- | :--- |
| 0    |     16.07      |


In [9]:
# Cell 25(code)
# 語法：逐列掃，每掃一列，將布林值用 ::int 轉成 0/1 → 加入 AVG 累積器(總和 += 該值、筆數 += 1) → 掃完最後一列(總和/筆數 = 0.1607)才是 AVG 動作整個完成。然後，才是把 0.1607 * 100.0 → ROUND 至小數點後第 2 位 → 掛別名成一列一欄(值 16.07、欄 churn_rate_pct (AS別名)) → 資料庫回傳給 pandas → pandas 包成 DataFrame (一列 x 一欄、列索引 0) → audit_h。
# 注意： ×100 和 ROUND 發生在資料庫端(這兩個動作寫在 SQL 指令裡)，pandas 收到的已是 16.07 這個完成計算的成品。
audit_h = pd.read_sql("""
    SELECT ROUND(100.0 * AVG((attrition_flag = 'Attrited Customer')::int), 2)
           AS churn_rate_pct
    FROM raw.bank_churners
""", engine)
audit_h

,churn_rate_pct
0,16.07


Cell 26(markdown)|判讀 + 稽核結論總表

**判讀**：<br/>
➡ 實測 = _____ % <br/>
1.必須 = 16.07。本案跨工具做了三次交叉驗證，確認此基準流失率用「同樣口徑(計算公式)、不同計算工具」都得出相同結果：M0 smoke test 、M1 .sql檔[H]、M1 notebook (本 cell)。<br/>
2.每個工具各自經過不同環節(載入資料、轉型別、布林值轉換、分組、相除)，但都得出一致結果，表示每個環節都通過檢驗，這就是交叉驗證的意義。若只有單一數字便無從確定對錯，但多個路徑都收斂出同一個值，「發生錯誤的機率」相對較低。<br/>
3.邊界：交叉驗證無法避免「口徑的定義錯誤」，若一開始就定義錯誤，三個工具會算出相同的錯誤數字。因為交叉驗證是「驗證執行是否遵循定義」，而「定義的正確性」是依靠「把計算公式在 M1 階段就寫明」，以供直接審視、檢討。<br/>
4.三種工具的檢驗：
|        工具        |          檔案名       |  計算路徑  |  驗證意義   |  
| :--- | :--- | :--- | :--- |
| M0 smoke test    |    notebooks/00_smoke_test.ipynb                 | DB 分組聚合 + pandas 相除(混合路徑) | 環境通、資料在、第一次拿到基準數字 | 
| M1 .sql 檔 [H]    |    sql/01_load_audit.sql                         | 純 DB 一行算完(AVG 慣用句) | 正式稽核紀錄的官方數字 |
| M1 notebook 本 cell    |    notebooks/01_audit_cleaning.ipynb       | 純 DB 同款查詢，經 pandas 執行 | notebook 載體與 .sql 檔口徑一致的證明 |


若三個工具都算出相同數字結果，表示：<br/>
(1)語意層正確：三條路徑對「什麼樣的情境屬於流失」的理解完全相同，核心邏輯一致 ⇒ 無語意解讀錯誤 (若因不同工具的不同計算邏輯造成數字不一致，要回頭檢視工具設計所依據的核心邏輯是否有誤)；<br/>
(2)資料庫內的資料載入後，在 M0、M1(現階段)都沒有發生異動。若資料載入當下就截斷，三種工具會一致算出同一個錯誤數字，此情境由 M0 的 COUNT 對照外部文件 10,127 來把關。原則：交叉驗證驗一致性，資料完整性檢驗驗正確性，兩種驗證各自檢查。

---

# 稽核結論總表(M1-B 清理決策的工作清單)

| # | 稽核項 | 實測結果 | 待決定的清理決策 |
|---|---|---|---|
| A | 主鍵唯一性 | 10127 = 10127、NULL = 0，唯一性成立 | clean 層以 clientnum 為 PRIMARY KEY |
| B | Unknown 占比 | edu 15.00% / income 10.98% / marital 7.40% | 傾向保留為獨立類別(剔除：損失樣本+引偏誤) |
| B | 樣本數過小的類別 | Platinum 20 人(0.20%)、Gold 116 人(1.15%) | 列入 M2/M3 「小樣本注意」清單 (切片時需標註「樣本不足，不單獨下結論」)；card 維度考慮合併非 Blue 的類別 |
| C | 可轉型稽核 | 14 欄 bad=0、NULL=0，實證數值欄資料乾淨 | 14 欄型別直接安全轉型，無需前置清洗 |
| D | 範圍合理性 | 全數落在業務邊界內；結構觀察:tenure min=13(無新戶)、trans_ct min=10(無呆卡)、額度上限是 34516 (M2 額度分布圖會出現尖峰，是產品規則的正常現象，非「資料異常」)、credit_limit/trans_amt 右偏 (max遠大於avg) | 不需做極端值處理 ( 因為 D 和 G 實測皆無不合理的極端值，而右偏的極端值如 額度 34516 是「真實存在的高價值客戶」，不是錯誤，所以要納入分析如 M5 的高價值挽留對象，刪掉反而會扭曲資料 )；<br/>將限制寫入 M2 方法段 (「新戶早期流失」這個經典高風險段在本資料中不存在，必須寫進 M2 的方法限制段)。資料分布右偏的欄位分箱：採等量分箱(NTILE)、不做等寬切分 |
| E | 共線性實證 | mismatch = 0，等式全表成立 | M4 排除 avg_open_to_buy，保留 total_revolving_bal |
| F | age × tenure 相關 | corr = 0.7889(社群流傳數字為 0.79-0.97 不等，常見引用 0.97；<br/>本實測證實數字在流傳區間的低點附近，落在分級決策的第二級) | M4 二擇一 (因為相關係數 0.79 的共線性放進迴歸仍會使係數不穩)；M2 可謹慎用 tenure，判讀時明示與年齡混淆的限制 (因兩者高度相關但存在獨立變異，tenure 非 age 的純影子) |
| G | 邏輯矛盾 | 六項全 0 | 無需矛盾清洗規則 |
| H | 基準流失率 |  16.07% (三路徑/工具交叉驗證的結果一致) | 全案指標口徑定案：流失率計算公式為「Attrited / 全體」，此後不再更改；之後若任何工具算出不同值，表示「該工具邏輯有誤」 |

> 本表填滿 = M1-A 階段完成。每一項的「待決定」欄就是 M1-B 階段的工作清單。


註 1：訊號 vs 雜訊：統計上，觀察到的流失率 = 真實流失傾向(訊號)+ 抽樣運氣造成的偏離(雜訊)。例如：拋擲一枚硬幣時，真實機率固定，但擲 20 次的結果可能大幅偏離、擲一萬次則非常貼近真實機率。因此，樣本越小，運氣成分(隨機性)占比越大。<br/>
註 2：樣本數過小的類別(如 card_category 的 Platinum 類別僅 20 人)，其切片格的比率受隨機波動主導。原因：持 Platinum 卡的客戶中，流失人數只要增減 1 人(微小變動)，流失率就波動 5 個百分點，這時觀察值幾乎全由隨機波動決定，訊號被淹沒；如此，觀察值由抽樣隨機性主導、而非真實流失傾向(訊號被雜訊淹沒)，故不可依此比率單獨下結論。<br/>
註 3：M2 額度分布圖會出現尖峰，是產品規則的正常現象：這家銀行的信用卡額度上限就是 34516，所以所有「本來可以給更高額度」的頂級客戶，額度全都被固定在這個天花板。於是有一大群人的額度會精確等於同一個數 (34516)。不是統計資料有誤，而是產品規則(信用卡額度上限固定)造成的資料聚集。判讀資料時，若看見資料分布的形狀特別，要先思考「是不是某條業務規則造成的」，然後才是懷疑資料是否有錯。

#### M1-B|清理結果驗收
- 完成稽核後，執行 python -m src.cleaning，讓資料清理由 `src/cleaning.py` 完成，然後下一個 cell 直接使用 pd.read_sql 將 cleaning.py 寫入的 clean 層資料讀出來，確認驗收成果是：(10127, 22) | 流失率: 16.07 % 以及 clean_df 的各欄位型別表。<br/>
- 清理規則與理由見 docs/decision_log.md [M1-1]~[M1-5]。<br/>
- 使用 cleaning.py 做資料清理的優點：由 src 清理、notebook 讀資料庫驗收，原則:清理邏輯在整個專案裡只存在一份(cleaning.py)，notebook 不重寫第二份。兩種資料清理的實作形式:
(1)需要用資料清理邏輯時 → import 來用:例如想在 notebook 示範「清理前後對比」，正確做法是： from src.cleaning import clean 然後餵資料給這個函式，不需要在 cell 裡重寫 drop 和 astype，因為在這裡重寫等於「讓同一套清理邏輯在兩份檔案分別存在兩個實作動作」；將來若需修改清理規則，若只改到其中一處，就會發生「兩個檔案存在不同的兩種邏輯」，造成 notebook 展示結果和 clean 層真實內容不一致 (且沒有機制提醒這個問題)。
(2)只需要看資料清理結果時 → 直接讀已用 python -m src.cleaning 清理完畢的資料庫成品:下面的驗收 cell 就是這種，直接讀 clean.customers，比起 import 的做法，這個是「只檢視資料清理 pipeline 的結果」，不會執行重複的資料清理。

In [10]:
# 資料清理結果驗收 cell
clean_df = pd.read_sql("SELECT * FROM clean.customers", engine)
print(clean_df.shape, "| 流失率:", round(clean_df["is_churned"].mean() * 100, 2), "%")
clean_df.dtypes

(10127, 22) | 流失率: 16.07 %


clientnum                     int64
attrition_flag                  str
customer_age                  int64
gender                          str
dependent_count               int64
education_level                 str
marital_status                  str
income_category                 str
card_category                   str
months_on_book                int64
total_relationship_count      int64
months_inactive_12_mon        int64
contacts_count_12_mon         int64
credit_limit                float64
total_revolving_bal         float64
avg_open_to_buy             float64
total_amt_chng_q4_q1        float64
total_trans_amt             float64
total_trans_ct                int64
total_ct_chng_q4_q1         float64
avg_utilization_ratio       float64
is_churned                     bool
dtype: object